<a href="https://colab.research.google.com/github/aritrasinha531-ai/silver-invention/blob/main/RNN_implementation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import pandas as pd

In [4]:
from google.colab import files
uploaded=files.upload()

Saving IMDB Dataset.csv to IMDB Dataset.csv


In [6]:
df=pd.read_csv("IMDB Dataset.csv")
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive
...,...,...
49995,I thought this movie did a down right good job...,positive
49996,"Bad plot, bad dialogue, bad acting, idiotic di...",negative
49997,I am a Catholic taught in parochial elementary...,negative
49998,I'm going to have to disagree with the previou...,negative


In [10]:
df.drop_duplicates(inplace=True)
df.shape

(49582, 2)

In [11]:
df

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive
...,...,...
49995,I thought this movie did a down right good job...,positive
49996,"Bad plot, bad dialogue, bad acting, idiotic di...",negative
49997,I am a Catholic taught in parochial elementary...,negative
49998,I'm going to have to disagree with the previou...,negative


In [15]:
# preprocessing
#1.converting to lowercase
df["review"]= df["review"].str.lower()

In [19]:
#2.removing urls
import re
def remove_urls(text):
  text=re.sub(r"http\s+","",text)#patterns,repl,string
  return  text
df["review"]=df["review"].apply(remove_urls)

In [20]:
#3.removing punctuation
def remove_punctuations(text):
  text=re.sub(r"[^A-Za-z0-9\s]","",text)
  return text
df["review"]=df["review"].apply(remove_punctuations)

In [21]:
#4.removing HTML
def remove_html(text):
  text=re.sub(r"<*?>","",text)
  return text
df["review"]=df["review"].apply(remove_html)

In [22]:
#5.removing stopwords
import nltk
nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")

from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

def remove_stopwords(text):
  tokens=word_tokenize(text)
  stop_words=stopwords.words("english")

  for word in tokens:
    if word in stop_words:
      text=text.replace(word,"")
  return text
df["review"]=df["review"].apply(remove_stopwords)
df.head()

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


,review,sentiment
0,e revewers nted wtchg 1 oz epode ll ho...,positive
1,wderful ltle producti br br filming techniqu...,positive
2,thought ths wderful wy spend tme o hot s...,positive
3,bsclly res fmly lttle boy jke thks res zom...,negative
4,petter mtte love time mey vully stunng fi...,positive


In [23]:
#6.stemming
from nltk.stem import PorterStemmer
def stemming(text):
  ps=PorterStemmer()
  stemmed_words=[]

  tokens=word_tokenize(text)
  for token in tokens:
    stemmed_tokens=ps.stem(token)
    stemmed_words.append(stemmed_tokens)
  return "".join(stemmed_words)
df["review"]=df["review"].apply(stemming)
df.head()

,review,sentiment
0,erevewntedwtchg1ozepodllhookyrghtexctlihppeneb...,positive
1,wderltleproductibrbrfilmtechniquunssumoldtimeb...,positive
2,thoughtthwderwyspendtmeohotsummerweekendsttngn...,positive
3,bscllirefmlilttleboyjkethkrezombclosetprentref...,negative
4,pettermttelovetimemeyvullistunngfilmwtchmrmtte...,positive


In [24]:
#7.encoding
from sklearn.preprocessing import LabelEncoder
le=LabelEncoder()
df["sentiment"]=le.fit_transform(df["sentiment"])
y=df["sentiment"]
y

,sentiment
0,1
1,1
2,1
3,0
4,1
...,...
49995,1
49996,0
49997,0
49998,0


In [25]:
#8.vectorization
from sklearn.feature_extraction.text import TfidfVectorizer
tf=TfidfVectorizer(max_features=5000)
x=tf.fit_transform(df["review"])
x

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 5008 stored elements and shape (49582, 5000)>

In [26]:
# datasets and dataloader
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42)

In [32]:
import torch
from torch.utils.data import TensorDataset,DataLoader
import scipy.sparse

# Convert x_train and x_test to dense arrays if they are sparse matrices
if isinstance(x_train, scipy.sparse.spmatrix):
    x_train = x_train.toarray()
if isinstance(x_test, scipy.sparse.spmatrix):
    x_test = x_test.toarray()

In [35]:
train_set=TensorDataset(torch.from_numpy(x_train).float(),torch.from_numpy(y_train.values).float())

In [36]:
test_set=TensorDataset(torch.from_numpy(x_test).float(),torch.from_numpy(y_test.values).float())
train_loader=DataLoader(train_set,batch_size=64,shuffle=True)
test_loader=DataLoader(test_set,shuffle=True,batch_size=64)

In [42]:
#build rnn model
import torch.nn as nn
import torch.optim as optim
class RNN(nn.Module):
  def __init__(self,input_size,hidden_size=128,num_layers=1):
    super(). __init__()
    self.hidden_size=hidden_size
    self.num_layers=num_layers

    #RNN layer
    self.rnn=nn.RNN(input_size,hidden_size,num_layers,batch_first=True)

    #fully connected layers
    self.fc=nn.Linear(hidden_size,1)

  def forward(self,x):
    h0=torch.zeros(self.num_layers,x.size(0),self.hidden_size)

    out,_=self.rnn(x,h0)

    out=self.fc(out[:,-1,:])
    return out
input_size=x_train.shape[1]

model=RNN(input_size)
criterion=nn.BCELoss()
optimizer=optim.Adam(model.parameters())

In [44]:
# train our RNN model
epochs=10
for epoch in range(epochs):
  model.train()
  for xb,yb in train_loader:
    optimizer.zero_grad()
    xb=xb.unsqueeze(1) # add singleton direction

    outputs=model.forward(xb)
    outputs=torch.sigmoid(outputs.squeeze())
    loss=criterion(outputs,yb)
    loss.backward()
    optimizer.step()
print(f"epoch={epoch+1}/{epochs} and loss={loss.item()}")

epoch=10/10 and loss=0.6520230174064636


In [47]:
# evaluate
model.eval()
with torch.no_grad():
  correct_vals=0
  total_vals=0
  for xb,yb in test_loader:
    xb=xb.unsqueeze(1)
    outputs=model.forward(xb)
    predicted=torch.sigmoid(outputs.squeeze()>0.5).float()

In [51]:
total_vals+=yb.size(0)
correct_vals+=(predicted==yb).sum().item()
print(f"accuracy={correct_vals/total_vals*100}")

accuracy=0.0


In [53]:
print("end of project")

end of project
